# Predict Progress

## Import Libraries

In [8]:
import pandas as pd
import pickle
import numpy as np
import onnxruntime as ort

In [9]:
# ==============================================================================
# GLOBAL METADATA & ONNX INITIALIZATION
# ==============================================================================

# 1. Load Metadata (Encoders & Feature List)
# We still load the pickle file JUST for the LabelEncoders and Feature Map.
with open('../../models/model_progress.pickle', 'rb') as f:
    data_metadata = pickle.load(f)

encoders = data_metadata['encoders']
feature_order = data_metadata['features']

# 2. Load ONNX Models
# Define the 9 numeric targets trained in the modeling notebook
targets_num = [
    'Weight_kg', 'Body_Fat_Percentage_y', 'Daily_Calories', 
    'Daily_Water_ml', 'Target_Protein_g', 'Target_Carbs_g', 'Target_Fat_g',
    'Limit_Sugar_g', 'Target_Fiber_g'
]

# Initialize ONNX Inference Sessions for each target
ort_sessions = {}
for target in targets_num:
    model_path = f"../../models/onnx/{target}.onnx"
    # Create an inference session for each model
    ort_sessions[target] = ort.InferenceSession(model_path)

print(f"✅ Loaded {len(ort_sessions)} ONNX models and metadata successfully.")

✅ Loaded 9 ONNX models and metadata successfully.


c:\Users\vince\anaconda3\envs\fitness_AI\lib\site-packages\sklearn\base.py:442: InconsistentVersionWarning: Trying to unpickle estimator LabelEncoder from version 1.8.0 when using version 1.7.2. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(


In [10]:
# ==============================================================================
# PREDICTION ENGINE (WEEKLY PROGRESS - ONNX VERSION)
# ==============================================================================

def prediksi_progres_lengkap(user_data, minggu_ke):
    """
    Predicts the user's physical status and nutritional needs using ONNX models.
    """

    # --- A. FEATURE ENGINEERING (Calculate Derived Metrics) ---
    tinggi_m = user_data['Height_cm'] / 100
    bmi_awal = round(user_data['Initial_Weight_kg'] / (tinggi_m ** 2), 2)
    
    # Determine Initial BMI Category
    if bmi_awal < 18.5: cat_bmi = 'Underweight'
    elif bmi_awal < 25: cat_bmi = 'Normal'
    elif bmi_awal < 30: cat_bmi = 'Overweight'
    else: cat_bmi = 'Obese'
    
    # --- B. DATA ENCODING (Text -> Number) ---
    gender_code = encoders['Gender'].transform([user_data['Gender']])[0]
    goal_code = encoders['Goal'].transform([user_data['Goal']])[0]
    level_code = encoders['level'].transform([user_data['level']])[0]
    bmi_cat_code = encoders['BMI_Category_x'].transform([cat_bmi])[0]

    # --- C. CONSTRUCT INPUT ARRAY (ONNX FORMAT) ---
    # Order must match the 'features' list from the modeling notebook
    input_row = [
        user_data['Age'], 
        gender_code, 
        user_data['Height_cm'], 
        user_data['Initial_Weight_kg'],
        bmi_awal, 
        bmi_cat_code, 
        user_data.get('Body_Fat_Category', 0), 
        user_data['Body_Fat_Percentage_x'],
        goal_code, 
        user_data['Workout_Frequency'], 
        user_data['Average_Duration_Minutes'], 
        level_code,
        user_data.get('Badminton', 0), 
        user_data.get('Football', 0), 
        user_data.get('Basketball', 0),
        user_data.get('Volleyball', 0), 
        user_data.get('Swim', 0),
        minggu_ke # Time variable
    ]
    
    # ONNX strictly requires float32 numpy arrays
    input_array = np.array([input_row], dtype=np.float32)
    
    # --- D. ONNX PREDICTION LOOP ---
    hasil = {}
    
    for target, sess in ort_sessions.items():
        # Get the expected input name for the ONNX node
        input_name = sess.get_inputs()[0].name
        
        # Run inference
        pred = sess.run(None, {input_name: input_array})[0]
        
        # Extract the scalar value and store it
        hasil[target] = float(pred[0][0])
        
    # --- E. POST-PROCESSING (Calculate missing targets) ---
    # Since BMI_Category_y and Meal_Frequency are no longer predicted by ML,
    # we calculate them manually based on the predicted results.
    
    # 1. Calculate Predicted BMI and Category
    pred_bmi = hasil['Weight_kg'] / (tinggi_m ** 2)
    hasil['BMI'] = pred_bmi
    
    if pred_bmi < 18.5: hasil['BMI_Category_y'] = 'Underweight'
    elif pred_bmi < 25: hasil['BMI_Category_y'] = 'Normal'
    elif pred_bmi < 30: hasil['BMI_Category_y'] = 'Overweight'
    else: hasil['BMI_Category_y'] = 'Obese'
        
    # 2. Calculate Meal Frequency based on predicted calories
    pred_cals = hasil['Daily_Calories']
    if pred_cals < 1500: hasil['Meal_Frequency'] = 3
    elif pred_cals < 2200: hasil['Meal_Frequency'] = 4
    else: hasil['Meal_Frequency'] = 5
            
    return hasil

In [11]:
# ==============================================================================
# SIMULATION & TESTING (EXECUTION)
# ==============================================================================

input_user = {
    'Age': 25, 
    'Gender': 'Male', 
    'Height_cm': 175, 
    'Initial_Weight_kg': 60,
    'Body_Fat_Category': 2, 
    'Body_Fat_Percentage_x': 15.0,
    'Goal': 'Muscle Gain', 
    'Workout_Frequency': 4, 
    'Average_Duration_Minutes': 60,
    'level': 'Beginner',
    'Badminton': 0, 
    'Football': 1, 
    'Basketball': 0, 
    'Volleyball': 0, 
    'Swim': 0
}

print(f"User: Pria, 25th, 60kg -> Goal: Muscle Gain")

for minggu in [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12]:
    
    output = prediksi_progres_lengkap(input_user, minggu_ke=minggu)
    
    print(f"\n{'='*10} MINGGU KE-{minggu} {'='*10}")
    
    print(f"[FISIK]")
    print(f"  Berat Badan     : {output['Weight_kg']:.2f} kg")
    print(f"  BMI             : {output['BMI']:.2f} ({output['BMI_Category_y']})")
    print(f"  Body Fat        : {output['Body_Fat_Percentage_y']:.1f}%")
    
    print(f"[NUTRISI HARIAN]")
    print(f"  Kalori          : {output['Daily_Calories']:.0f} kkal")
    print(f"  Air Minum       : {output['Daily_Water_ml']:.0f} ml")
    print(f"  Gula (Limit)    : {output['Limit_Sugar_g']:.1f} g")
    print(f"  Meal Freq       : {output['Meal_Frequency']}x / hari")
    
    print(f"[MAKRO NUTRISI]")
    print(f"  Protein         : {output['Target_Protein_g']:.1f} g")
    print(f"  Karbo           : {output['Target_Carbs_g']:.1f} g")
    print(f"  Lemak           : {output['Target_Fat_g']:.1f} g")
    print(f"  Serat           : {output['Target_Fiber_g']:.1f} g")

User: Pria, 25th, 60kg -> Goal: Muscle Gain

========== MINGGU KE-1 ==========
[FISIK]
  Berat Badan     : 60.26 kg
  BMI             : 19.68 (Normal)
  Body Fat        : 14.8%
[NUTRISI HARIAN]
  Kalori          : 2712 kkal
  Air Minum       : 2691 ml
  Gula (Limit)    : 67.0 g
  Meal Freq       : 5x / hari
[MAKRO NUTRISI]
  Protein         : 201.8 g
  Karbo           : 333.2 g
  Lemak           : 60.2 g
  Serat           : 37.7 g

========== MINGGU KE-2 ==========
[FISIK]
  Berat Badan     : 60.38 kg
  BMI             : 19.72 (Normal)
  Body Fat        : 14.6%
[NUTRISI HARIAN]
  Kalori          : 2712 kkal
  Air Minum       : 2691 ml
  Gula (Limit)    : 67.0 g
  Meal Freq       : 5x / hari
[MAKRO NUTRISI]
  Protein         : 201.8 g
  Karbo           : 333.2 g
  Lemak           : 60.2 g
  Serat           : 37.7 g

========== MINGGU KE-3 ==========
[FISIK]
  Berat Badan     : 60.51 kg
  BMI             : 19.76 (Normal)
  Body Fat        : 14.5%
[NUTRISI HARIAN]
  Kalori          : 2712